# 03 - Building an Agent, UX and Human-in-the-Loop (HITL)

A small tool-calling agent, paused before it runs a tool so a human can approve it.

In [ ]:
import os
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver
from langchain_openai import ChatOpenAI

load_dotenv()

## Building an Agent with LangGraph

A tool, an LLM bound to that tool, and a graph that loops between "agent" and "tools".

In [ ]:
@tool
def get_order_status(order_id: str) -> str:
    """Look up the status of an order."""
    return f"Order {order_id} is out for delivery."

llm = ChatOpenAI(model="gpt-4o-mini")
llm_with_tools = llm.bind_tools([get_order_status])

def agent_node(state):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

builder = StateGraph(MessagesState)
builder.add_node("agent", agent_node)
builder.add_node("tools", ToolNode([get_order_status]))
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition)
builder.add_edge("tools", "agent")

## UX and Human-in-the-Loop (HITL)

`interrupt_before=["tools"]` pauses the graph right before the tool node runs,
so a human gets a chance to approve/reject the tool call.

In [ ]:
memory = MemorySaver()
agent_app = builder.compile(checkpointer=memory, interrupt_before=["tools"])

config = {"configurable": {"thread_id": "hitl-thread-1"}}

state = agent_app.invoke({"messages": [HumanMessage(content="What's the status of order 4521?")]}, config)
print("Pending tool call:", state["messages"][-1].tool_calls)

Graph is paused. A human reviews the pending tool call above, then approves by
resuming the graph with no new input (`None`).

In [ ]:
final_state = agent_app.invoke(None, config)
final_state["messages"][-1].content